# 03 - Analysis (tables, figures, 6/19 fixes, geometry)

Analysis-only: reads the unified DBs from notebooks 01/02 and regenerates every report artifact plus the 6/19 corrections. **No GPU/simulator needed.**

- **Tables 2-5** (controlled slice): SR, recovery/degradation, detector metrics, taxonomy (best-of-three step-config collapse).
- **6/19 section 1.3**: per-suite-stratified AUC reported alongside pooled.
- **Tables 8-9** (LIBERO-PRO): baseline SR + per-suite/per-DOF detector AUC.
- **Table 6**: PCP three-way eval.
- **Geometry (net-new)**: Sarle bimodality, PCA isotropy + Marchenko-Pastur, directional var_par/var_lat, online-feature gating, length confound.
- **Cross-model** pi0.5 vs SmolVLA transfer + a *numbers-changed-vs-report* CSV.

## 1. Config + helpers

In [ ]:
# === Analysis config (runs anywhere the DBs / ahats are reachable) ===========
import os
try:
    from google.colab import drive
    drive.mount('/content/drive'); DRIVE = '/content/drive/MyDrive'
except Exception:
    DRIVE = os.path.expanduser('~')

FINAL_ROOT  = os.path.join(DRIVE, 'cs159-sp26-final')
RESULTS_DIR = os.path.join(FINAL_ROOT, 'results')
AHATS_DIR   = os.path.join(RESULTS_DIR, 'ahats')
FIGURES_DIR = os.path.join(FINAL_ROOT, 'figures')
TABLES_DIR  = os.path.join(FINAL_ROOT, 'tables')
SLICE_DB = os.path.join(RESULTS_DIR, 'rollouts_final.db')
PRO_DB   = os.path.join(RESULTS_DIR, 'rollouts_pro.db')
QC_DB    = os.path.join(RESULTS_DIR, 'qc.db')
for _d in (FIGURES_DIR, TABLES_DIR):
    os.makedirs(_d, exist_ok=True)
print('FINAL_ROOT =', FINAL_ROOT)


In [ ]:
# === Shared analysis helpers ================================================
import sqlite3, json, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
try:
    from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
except Exception:
    roc_auc_score = average_precision_score = f1_score = None

pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)

# Known report (Aviato Dynamics) headline numbers, for the delta-vs-report CSV.
REPORT_NUMBERS = {
    'slice_SR_vanilla': 76.2,
    'slice_SR_extra_steps': None,
    'slice_SR_pnp_uncertainty_only': 90.0,
    'slice_SR_pnp_refinement': None,
}
DELTAS = []   # rows: metric, report_value, recomputed_value


def load_rollouts(path):
    if not os.path.exists(path):
        print('MISSING:', path); return pd.DataFrame()
    con = sqlite3.connect(path)
    df = pd.read_sql_query('SELECT * FROM rollouts', con); con.close()
    return df


def detector_metrics(score, fail):
    score = np.asarray(score, float); fail = np.asarray(fail, int)
    m = np.isfinite(score); score, fail = score[m], fail[m]
    out = {'n': int(len(score)), 'n_fail': int(fail.sum())}
    if roc_auc_score is None or len(np.unique(fail)) < 2 or len(score) < 3:
        out.update(roc_auc=np.nan, pr_auc=np.nan, spearman=np.nan, f1=np.nan, tau=np.nan)
        return out
    out['roc_auc'] = roc_auc_score(fail, score)
    out['pr_auc'] = average_precision_score(fail, score)
    out['spearman'] = stats.spearmanr(score, fail).correlation
    best_f1, best_t = -1.0, np.nan
    for t in np.quantile(score, np.linspace(0.05, 0.95, 19)):
        f = f1_score(fail, (score >= t).astype(int), zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    out['f1'], out['tau'] = best_f1, best_t
    return out


def stratified_auc(df, score_col, by='suite', fail_col='fail'):
    """6/19 section 1.3 fix: AUC computed within each suite, then averaged."""
    aucs = []
    for s, g in df.groupby(by):
        if roc_auc_score is None or g[fail_col].nunique() < 2:
            continue
        aucs.append(roc_auc_score(g[fail_col], g[score_col]))
    return float(np.mean(aucs)) if aucs else np.nan, len(aucs)


def savefig(name):
    p = os.path.join(FIGURES_DIR, name)
    plt.tight_layout(); plt.savefig(p, dpi=150); plt.show()
    print('wrote', p)


def savetable(df, name):
    p = os.path.join(TABLES_DIR, name)
    df.to_csv(p, index=False); print('wrote', p)


## 2. Tables 2-5 (controlled slice)

In [ ]:
# === Tables 2-5: controlled 80-ep slice =====================================
slice_df = load_rollouts(SLICE_DB)
if not slice_df.empty:
    slice_df['fail'] = 1 - slice_df['success']
    keys = ['policy_model', 'method', 'suite', 'task_idx', 'episode_idx', 'init_state_hash']

    # Best-of-three step-config collapse (success = max over the 3 P&P step configs).
    collapsed = (slice_df.groupby(keys, dropna=False)
                 .agg(success=('success', 'max'),
                      u_mean_episode=('u_mean_episode', 'mean'),
                      n_steps=('n_steps', 'mean')).reset_index())

    # Table 2: SR by method/model.
    t2 = (collapsed.groupby(['policy_model', 'method'])
          .agg(SR=('success', 'mean'), n=('success', 'size')).reset_index())
    t2['SR'] = (100 * t2['SR']).round(1)
    print('=== Table 2: success rate by method ===')
    print(t2.to_string(index=False))
    savetable(t2, 'table2_success_rate.csv')
    for _, r in t2.iterrows():
        key = f"slice_SR_{r['method']}"
        if r['policy_model'] == 'pi05' and key in REPORT_NUMBERS:
            DELTAS.append(dict(metric=f"{key} (pi05)", report_value=REPORT_NUMBERS[key],
                               recomputed_value=r['SR']))

    # Table 3: recovery / degradation vs vanilla.
    t3rows = []
    for model, md_ in collapsed.groupby('policy_model'):
        piv = md_.pivot_table(index=['suite', 'task_idx', 'episode_idx', 'init_state_hash'],
                              columns='method', values='success')
        if 'vanilla' not in piv:
            continue
        v = piv['vanilla']
        for method in [m for m in piv.columns if m != 'vanilla']:
            mth = piv[method]; mask = v.notna() & mth.notna()
            vv, mm = v[mask], mth[mask]
            t3rows.append(dict(policy_model=model, method=method, n=int(mask.sum()),
                               recovery=int(((vv == 0) & (mm == 1)).sum()),
                               degradation=int(((vv == 1) & (mm == 0)).sum())))
    t3 = pd.DataFrame(t3rows)
    print('\n=== Table 3: recovery / degradation vs vanilla ===')
    print(t3.to_string(index=False))
    savetable(t3, 'table3_recovery_degradation.csv')

    # Table 4: detector metrics per step config (pnp_uncertainty_only, U->failure).
    det = slice_df[slice_df['method'] == 'pnp_uncertainty_only'].copy()
    t4rows = []
    for (model, cfg), g in det.groupby(['policy_model', 'pnp_step_indices']):
        m = detector_metrics(g['u_mean_episode'], g['fail'])
        t4rows.append(dict(policy_model=model, step_config=cfg, **m))
    t4 = pd.DataFrame(t4rows)
    print('\n=== Table 4: P&P detector metrics per step config ===')
    print(t4.to_string(index=False))
    savetable(t4, 'table4_detector_metrics.csv')

    # Table 5: median-split uncertainty taxonomy (low/high U x success/fail).
    t5rows = []
    for model, g in det.groupby('policy_model'):
        gg = g.dropna(subset=['u_mean_episode'])
        if gg.empty:
            continue
        med = gg['u_mean_episode'].median()
        gg = gg.assign(U=np.where(gg['u_mean_episode'] >= med, 'high_U', 'low_U'))
        tab = pd.crosstab(gg['U'], np.where(gg['success'] == 1, 'success', 'fail'))
        print(f'\n=== Table 5: taxonomy ({model}) median U={med:.4f} ===')
        print(tab)
        tab2 = tab.reset_index(); tab2.insert(0, 'policy_model', model)
        t5rows.append(tab2)
    if t5rows:
        savetable(pd.concat(t5rows, ignore_index=True), 'table5_taxonomy.csv')
else:
    print('No slice data yet -- run notebook 01 first.')


## 3. Stratified vs pooled detector AUC (6/19 section 1.3)

In [ ]:
# === 6/19 section 1.3: stratified (per-suite) AUC vs pooled AUC ==============
strat_rows = []
for label, path, method in [('slice', SLICE_DB, 'pnp_uncertainty_only'),
                            ('pro', PRO_DB, 'pnp_uncertainty_only')]:
    df = load_rollouts(path)
    if df.empty:
        continue
    df = df[df['method'] == method].copy()
    df['fail'] = 1 - df['success']
    df = df.dropna(subset=['u_mean_episode'])
    for model, g in df.groupby('policy_model'):
        if roc_auc_score is None or g['fail'].nunique() < 2:
            continue
        pooled = roc_auc_score(g['fail'], g['u_mean_episode'])      # old (pooled) method
        strat, n_suites = stratified_auc(g, 'u_mean_episode')       # 6/19 fix
        strat_rows.append(dict(dataset=label, policy_model=model, pooled_auc=round(pooled, 4),
                               stratified_auc=round(strat, 4), n_suites=n_suites,
                               delta=round(pooled - strat, 4)))
        DELTAS.append(dict(metric=f'{label} detector AUC pooled ({model})',
                           report_value='pooled', recomputed_value=round(pooled, 4)))
        DELTAS.append(dict(metric=f'{label} detector AUC stratified ({model})',
                           report_value='per-suite-mean', recomputed_value=round(strat, 4)))
strat = pd.DataFrame(strat_rows)
print('=== Pooled vs per-suite-stratified detector AUC ===')
print(strat.to_string(index=False) if not strat.empty else 'no detector rows')
if not strat.empty:
    savetable(strat, 'stratified_vs_pooled_auc.csv')


## 4. Tables 8-9 (LIBERO-PRO + per-DOF)

In [ ]:
# === Tables 8-9: LIBERO-PRO aggregate + per-DOF ============================
def load_perdof(path, method='pnp_uncertainty_only'):
    if not os.path.exists(path):
        return pd.DataFrame()
    con = sqlite3.connect(path)
    dims = ", ".join([f"AVG(e.u_d{i}) ud{i}" for i in range(7)])
    df = pd.read_sql_query(
        "SELECT r.rollout_id, r.suite, r.task_idx, r.episode_idx, r.success, r.n_steps, "
        "r.policy_model, AVG(e.u_mean) u_mean, " + dims + " "
        "FROM rollouts r JOIN pnp_euler_steps e ON r.rollout_id=e.rollout_id "
        "WHERE r.method=? GROUP BY r.rollout_id", con, params=(method,))
    con.close()
    return df

pro_df = load_rollouts(PRO_DB)
if not pro_df.empty:
    pro_df['fail'] = 1 - pro_df['success']
    # Table 8: baseline SR + per-suite detector AUC.
    base = pro_df[pro_df['method'] == 'vanilla']
    unc = pro_df[pro_df['method'] == 'pnp_uncertainty_only'].dropna(subset=['u_mean_episode'])
    t8rows = []
    for model in pro_df['policy_model'].dropna().unique():
        for suite in sorted(pro_df['suite'].unique()):
            b = base[(base.policy_model == model) & (base.suite == suite)]
            u = unc[(unc.policy_model == model) & (unc.suite == suite)]
            auc = (roc_auc_score(u['fail'], u['u_mean_episode'])
                   if roc_auc_score and u['fail'].nunique() > 1 else np.nan)
            t8rows.append(dict(policy_model=model, suite=suite,
                               baseline_SR=round(100 * b['success'].mean(), 1) if len(b) else np.nan,
                               n=len(u), detector_auc=round(auc, 4) if auc == auc else np.nan))
    t8 = pd.DataFrame(t8rows)
    print('=== Table 8: LIBERO-PRO baseline SR + per-suite detector AUC ===')
    print(t8.to_string(index=False))
    savetable(t8, 'table8_pro_aggregate.csv')

    # Table 9: 7-dim vs pos+grip subset AUC + per-DOF AUC.
    pdf = load_perdof(PRO_DB)
    if not pdf.empty:
        pdf['fail'] = 1 - pdf['success']
        pos_grip = ['ud0', 'ud1', 'ud2', 'ud6']
        all_dims = [f'ud{i}' for i in range(7)]
        pdf['score_full'] = np.sqrt((pdf[all_dims] ** 2).sum(axis=1))
        pdf['score_posgrip'] = np.sqrt((pdf[pos_grip] ** 2).sum(axis=1))
        t9rows = []
        for model, g in pdf.groupby('policy_model'):
            full_s, _ = stratified_auc(g, 'score_full')
            pg_s, _ = stratified_auc(g, 'score_posgrip')
            row = dict(policy_model=model, auc_7dim=round(full_s, 4), auc_pos_grip=round(pg_s, 4))
            for i in range(7):
                a, _ = stratified_auc(g, f'ud{i}')
                row[f'auc_ud{i}'] = round(a, 4)
            t9rows.append(row)
        t9 = pd.DataFrame(t9rows)
        print('\n=== Table 9: per-DOF detector AUC (per-suite averaged) ===')
        print(t9.to_string(index=False))
        savetable(t9, 'table9_per_dof.csv')
else:
    print('No PRO data yet -- run notebook 01 PRO driver first.')


## 5. Table 6 (PCP)

In [ ]:
# === Table 6: PCP three-way eval ===========================================
if os.path.exists(QC_DB):
    con = sqlite3.connect(QC_DB)
    try:
        qe = pd.read_sql_query('SELECT * FROM qc_eval', con)
    except Exception:
        qe = pd.DataFrame()
    con.close()
    if not qe.empty:
        name_map = {-1.0: 'vanilla', 0.0: 'PnP-only (lambda=0)', 3.0: 'PCP (lambda=3)'}
        qe['arm'] = qe['lambda'].map(lambda x: name_map.get(x, f'lambda={x}'))
        t6 = (qe.groupby('arm').agg(SR=('success', 'mean'), n=('success', 'size')).reset_index())
        t6['SR'] = (100 * t6['SR']).round(1)
        print('=== Table 6: PCP three-way (LIBERO-PRO hard tasks) ===')
        print(t6.to_string(index=False))
        savetable(t6, 'table6_pcp.csv')
        for _, r in t6.iterrows():
            DELTAS.append(dict(metric=f"PCP {r['arm']} SR", report_value=None, recomputed_value=r['SR']))
    else:
        print('qc_eval empty -- run notebook 02 eval first.')
else:
    print('No qc.db -- run notebook 02 first.')


## 6. Geometry of uncertainty

In [ ]:
# === Geometry of uncertainty (net-new, 6/19 sections 2.2-2.6) ===============
def _sarle_bc(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    n = len(x)
    if n < 4:
        return np.nan
    g = stats.skew(x); k = stats.kurtosis(x, fisher=True)
    denom = k + 3.0 * (n - 1) ** 2 / ((n - 2) * (n - 3))
    return (g ** 2 + 1.0) / denom if denom else np.nan


def load_ahats_index(ahats_dir, ref_df):
    """Map each persisted ahats npz (named <rollout_id>.npz) to its rollout row."""
    if not os.path.isdir(ahats_dir):
        print('No ahats dir:', ahats_dir); return []
    ref = ref_df.set_index('rollout_id') if 'rollout_id' in ref_df else None
    items = []
    for fp in glob.glob(os.path.join(ahats_dir, '*.npz')):
        rid = os.path.splitext(os.path.basename(fp))[0]
        if ref is None or rid not in ref.index:
            continue
        items.append((fp, ref.loc[rid]))
    return items


pro_unc = pro_df[pro_df['method'] == 'pnp_uncertainty_only'].copy() if not pro_df.empty else pd.DataFrame()
ah_items = load_ahats_index(AHATS_DIR, pro_unc) if not pro_unc.empty else []
print(f'ahats episodes available: {len(ah_items)}')

if ah_items:
    bc_rows, dir_rows = [], []
    suite_vecs = {}    # suite -> list of per-episode mean-action vectors (for PCA)
    for fp, row in ah_items:
        npz = np.load(fp)
        # each entry: (K, B, chunk, adim); pool deviations across stored steps.
        devs, means, parr, latv = [], [], [], []
        for key in npz.files:
            A = npz[key]                       # (K,B,chunk,adim)
            A = A.reshape(A.shape[0], -1)      # (K, D)
            m = A.mean(0)
            d = A - m                          # (K, D)
            devs.append(d.reshape(-1))
            means.append(m)
            nrm = np.linalg.norm(m) + 1e-8
            u = m / nrm
            proj = d @ u                       # (K,)  parallel component
            parr.append(float((proj ** 2).mean()))
            latv.append(float((np.sum(d ** 2, axis=1) - proj ** 2).mean() / max(d.shape[1] - 1, 1)))
        bc = _sarle_bc(np.concatenate(devs)) if devs else np.nan
        bc_rows.append(dict(suite=row['suite'], success=int(row['success']), fail=int(1 - row['success']),
                            bc=bc, var_par=float(np.mean(parr)), var_lat=float(np.mean(latv)),
                            par_lat_ratio=float(np.mean(parr) / (np.mean(latv) + 1e-8))))
        suite_vecs.setdefault(row['suite'], []).append(np.mean(means, axis=0))

    bc_df = pd.DataFrame(bc_rows)

    # 2.2 Multimodality: BC by outcome + AUC(BC -> failure), per suite then averaged.
    bc_summary = (bc_df.groupby('suite')
                  .apply(lambda g: pd.Series({
                      'BC|succ': g.loc[g.success == 1, 'bc'].mean(),
                      'BC|fail': g.loc[g.fail == 1, 'bc'].mean(),
                      'n': len(g)})).reset_index())
    bc_auc, _ = stratified_auc(bc_df.dropna(subset=['bc']), 'bc', fail_col='fail')
    print('=== Multimodality (Sarle BC, 0.555 = bimodal threshold) ===')
    print(bc_summary.to_string(index=False))
    print(f'AUC(BC -> failure), per-suite mean = {bc_auc:.4f}')
    savetable(bc_summary, 'geometry_multimodality_bc.csv')

    # 2.4 Directional: Tier-2 var_par / var_lat ratio + AUC.
    ratio_auc, _ = stratified_auc(bc_df.dropna(subset=['par_lat_ratio']), 'par_lat_ratio', fail_col='fail')
    dir_summary = (bc_df.groupby('suite')
                   .agg(var_par=('var_par', 'mean'), var_lat=('var_lat', 'mean'),
                        ratio=('par_lat_ratio', 'mean')).reset_index())
    print('\n=== Directional (Tier-2 var_par/var_lat) ===')
    print(dir_summary.to_string(index=False))
    print(f'AUC(par/lat ratio -> failure), per-suite mean = {ratio_auc:.4f}')
    savetable(dir_summary, 'geometry_directional.csv')

    # 2.3 PCA isotropy on per-episode mean-action vectors (Marchenko-Pastur reference).
    iso_rows = []
    for suite, vecs in suite_vecs.items():
        X = np.vstack(vecs)
        if X.shape[0] < 3:
            continue
        Xc = X - X.mean(0)
        N, D = Xc.shape
        sv = np.linalg.svd(Xc, compute_uv=False)
        ev = (sv ** 2) / max(N - 1, 1)
        pc1_frac = float(ev[0] / ev.sum()) if ev.sum() else np.nan
        sigma2 = float(ev.mean())
        ratio = D / N
        mp_max = sigma2 * (1 + np.sqrt(ratio)) ** 2      # MP upper edge (noise reference)
        iso_rows.append(dict(suite=suite, N=N, D=D, pc1_frac=round(pc1_frac, 4),
                             lambda_max=round(float(ev[0]), 6), mp_upper_edge=round(mp_max, 6),
                             above_mp=bool(ev[0] > mp_max)))
    iso = pd.DataFrame(iso_rows)
    print('\n=== PCA isotropy (PC1 fraction vs Marchenko-Pastur) ===')
    print(iso.to_string(index=False))
    savetable(iso, 'geometry_pca_isotropy.csv')
else:
    print('No ahats persisted -- run notebook 01 PRO uncertainty pass (record_per_iteration=True).')

# 2.5 Online features + 2.6 length confound (from pnp_euler_steps; no ahats needed).
if os.path.exists(PRO_DB):
    con = sqlite3.connect(PRO_DB)
    steps = pd.read_sql_query(
        "SELECT e.rollout_id, e.chunk_idx, AVG(e.u_mean) u "
        "FROM pnp_euler_steps e JOIN rollouts r ON r.rollout_id=e.rollout_id "
        "WHERE r.method='pnp_uncertainty_only' GROUP BY e.rollout_id, e.chunk_idx", con)
    meta = pd.read_sql_query(
        "SELECT rollout_id, suite, success, n_steps, u_mean_episode FROM rollouts "
        "WHERE method='pnp_uncertainty_only'", con)
    con.close()
    if not steps.empty and not meta.empty:
        meta = meta.set_index('rollout_id')
        thr = steps['u'].quantile(0.75)
        feat_rows = []
        for rid, g in steps.groupby('rollout_id'):
            if rid not in meta.index:
                continue
            seq = g.sort_values('chunk_idx')['u'].to_numpy()
            ema = seq[0]
            for v in seq[1:]:
                ema = 0.7 * ema + 0.3 * v
            run = np.maximum.accumulate(np.cumsum(seq) / (np.arange(len(seq)) + 1))
            m = meta.loc[rid]
            feat_rows.append(dict(
                rollout_id=rid, suite=m['suite'], fail=int(1 - m['success']),
                n_steps=int(m['n_steps']),
                f_instant=float(np.mean(seq)),
                f_ema=float(ema),
                f_dev_runmean=float(np.max(seq - run)),
                f_cum_highU=float(np.sum(seq >= thr)),
                f_chunk_idx=float(len(seq)),
                f_progress=float(m['n_steps']),
                f_early=float(np.mean(seq[:2])),
            ))
        F = pd.DataFrame(feat_rows)
        online_feats = ['f_instant', 'f_ema', 'f_dev_runmean', 'f_cum_highU',
                        'f_chunk_idx', 'f_progress', 'f_early']
        on_rows = [dict(feature=f, auc=round(stratified_auc(F.dropna(subset=[f]), f, fail_col='fail')[0], 4))
                   for f in online_feats]
        on_df = pd.DataFrame(on_rows)
        print('\n=== Online feature gating: per-suite-mean AUC(feature -> failure) ===')
        print(on_df.to_string(index=False))
        savetable(on_df, 'geometry_online_features.csv')

        # Length confound: r(U, episode length) + early-window vs full AUC.
        r_ul = stats.pearsonr(F['f_instant'], F['n_steps'])[0] if len(F) > 2 else np.nan
        auc_full, _ = stratified_auc(F, 'f_instant', fail_col='fail')
        auc_early, _ = stratified_auc(F, 'f_early', fail_col='fail')
        print(f'\n=== Length confound ===\nr(U, length) = {r_ul:.4f}  '
              f'AUC(full U)={auc_full:.4f}  AUC(early-window U)={auc_early:.4f}')
        savetable(pd.DataFrame([dict(r_U_length=round(r_ul, 4), auc_full_U=round(auc_full, 4),
                                     auc_early_U=round(auc_early, 4))]),
                  'geometry_length_confound.csv')


## 7. Cross-model transfer + delta-vs-report

In [ ]:
# === Cross-model detector transfer + delta-vs-report CSV ====================
cm_rows = []
for label, path in [('slice', SLICE_DB), ('pro', PRO_DB)]:
    df = load_rollouts(path)
    if df.empty:
        continue
    df = df[df['method'] == 'pnp_uncertainty_only'].dropna(subset=['u_mean_episode']).copy()
    df['fail'] = 1 - df['success']
    for model, g in df.groupby('policy_model'):
        auc, n = stratified_auc(g, 'u_mean_episode')
        cm_rows.append(dict(dataset=label, policy_model=model, detector_auc=round(auc, 4), n_suites=n))
cm = pd.DataFrame(cm_rows)
print('=== Cross-model detector AUC (per-suite averaged) ===')
print(cm.to_string(index=False) if not cm.empty else 'no data')
if not cm.empty:
    savetable(cm, 'crossmodel_detector_auc.csv')
    piv = cm.pivot_table(index='dataset', columns='policy_model', values='detector_auc')
    if {'pi05', 'smolvla'}.issubset(piv.columns):
        ax = piv.plot(kind='bar', figsize=(6, 4))
        ax.set_ylabel('per-suite-mean detector AUC'); ax.set_title('pi0.5 vs SmolVLA P&P detector')
        savefig('crossmodel_detector_auc.png')

delta_df = pd.DataFrame(DELTAS)
print('\n=== Numbers changed vs report ===')
print(delta_df.to_string(index=False) if not delta_df.empty else 'no deltas recorded')
if not delta_df.empty:
    savetable(delta_df, 'numbers_changed_vs_report.csv')
print('\nAnalysis complete. Tables ->', TABLES_DIR, ' Figures ->', FIGURES_DIR)
